# Aula 2 — Notebook 4  
## Revisão, limites e evoluções

Neste notebook, você vai consolidar um fluxo completo de investigação de incidentes com agentes e ferramentas reais.
A ideia central é separar duas camadas:
- uma camada **probabilística** (o agente interpreta contexto e propõe caminhos);
- uma camada **determinística** (regras de validação decidem se a proposta pode avançar).

O objetivo desta aula é aprender a construir um processo confiável para análise técnica, e não gerar patch automaticamente.
Ao final, você terá um artefato estruturado que poderá servir como base para uma etapa posterior de implementação.

Uma proposta deve ser bloqueada quando:

- não cita evidências consultadas;
- aponta arquivos que não existem no `target_project`;
- sugere alteração fora do escopo do incidente;
- não inclui plano de testes;
- tenta executar automaticamente mudanças sem revisão humana.

Durante a leitura do notebook, observe como cada célula reduz um tipo de risco:
1. risco de contexto incompleto (busca de evidências);
2. risco de saída inconsistente (contrato JSON);
3. risco de ação insegura (validação determinística + revisão humana).

In [1]:
from pathlib import Path
import os
import json
import textwrap
import subprocess
import sys
from datetime import datetime

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

candidatos_target_project = [
    PROJECT_ROOT / "target_project",
    PROJECT_ROOT.parent / "target_project",
]
TARGET_PROJECT_DIR = next((p for p in candidatos_target_project if p.exists()), candidatos_target_project[0])

if not TARGET_PROJECT_DIR.exists():
    caminhos = "\n".join([f"- {p.as_posix()}" for p in candidatos_target_project])
    raise FileNotFoundError(
        "Não foi possível localizar a pasta target_project. Caminhos tentados:\n" + caminhos
    )

print(f"TARGET_PROJECT_DIR resolvido para: {TARGET_PROJECT_DIR.as_posix()}")

TARGET_PROJECT_DIR resolvido para: c:/Users/cstefano/Desktop/Carlos/Alura/Skills and Go/Repositório/alura-skills-and-go-agentic-engineering/target_project


## Preparação do ambiente e importações

Antes de qualquer lógica de agente, precisamos resolver dois pontos básicos:
- acesso a pastas e arquivos do projeto;
- utilitários para serializar dados, executar testes e controlar tempo de execução.

As células anteriores já trouxeram essas importações e definiram caminhos centrais
(`PROJECT_ROOT`, `DATA_DIR`, `TARGET_PROJECT_DIR`, `OUTPUT_DIR`) para manter referências consistentes no notebook inteiro.

Esse cuidado evita erros comuns de caminho relativo, principalmente quando o notebook é executado a partir de diretórios diferentes.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXTENSOES_INDEXADAS = {".md", ".txt", ".log", ".py", ".json", ".toml"}


def ler_texto(caminho):
    return Path(caminho).read_text(encoding="utf-8")


def caminho_relativo_seguro(caminho):
    caminho = Path(caminho)
    bases = [PROJECT_ROOT, PROJECT_ROOT.parent]
    for base in bases:
        try:
            return caminho.relative_to(base).as_posix()
        except ValueError:
            continue
    return caminho.as_posix()


def listar_arquivos_recursivamente(raiz, extensoes=EXTENSOES_INDEXADAS):
    arquivos = []
    for caminho in Path(raiz).rglob("*"):
        if caminho.is_file() and caminho.suffix in extensoes:
            arquivos.append(caminho)
    return sorted(arquivos)


def montar_corpus(data_dir=DATA_DIR, target_project_dir=TARGET_PROJECT_DIR):
    arquivos = listar_arquivos_recursivamente(data_dir) + listar_arquivos_recursivamente(target_project_dir)
    documentos = []
    for caminho in arquivos:
        relativo = caminho_relativo_seguro(caminho)
        documentos.append({
            "path": relativo,
            "content": ler_texto(caminho),
            "source_type": "target_project" if "target_project" in relativo else "evidence",
        })
    return documentos


def criar_indice_tfidf(documentos):
    vectorizer = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2))
    matriz = vectorizer.fit_transform([doc["content"] for doc in documentos])
    return {"vectorizer": vectorizer, "matrix": matriz, "documents": documentos}


def buscar_no_indice(indice, consulta, k=5):
    consulta_vetor = indice["vectorizer"].transform([consulta])
    scores = cosine_similarity(consulta_vetor, indice["matrix"])[0]
    ordenados = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:k]
    resultados = []
    for idx, score in ordenados:
        doc = indice["documents"][idx]
        resultados.append({
            "path": doc["path"],
            "score": round(float(score), 4),
            "source_type": doc["source_type"],
            "preview": doc["content"][:1200],
        })
    return resultados


def listar_tickets():
    return [caminho_relativo_seguro(p) for p in sorted((DATA_DIR / "tickets").glob("*.md"))]


def carregar_arquivo_relativo(caminho_relativo):
    caminho = PROJECT_ROOT / caminho_relativo
    if not caminho.exists():
        caminho_alternativo = PROJECT_ROOT.parent / caminho_relativo
        if caminho_alternativo.exists():
            caminho = caminho_alternativo
    if not caminho.exists():
        return json.dumps({"erro": f"Arquivo não encontrado: {caminho_relativo}"}, ensure_ascii=False)
    return caminho.read_text(encoding="utf-8")


def listar_arquivos_do_target_project():
    arquivos = listar_arquivos_recursivamente(TARGET_PROJECT_DIR)
    return [caminho_relativo_seguro(p) for p in arquivos]


def executar_testes_target_project():
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=TARGET_PROJECT_DIR,
        capture_output=True,
        text=True,
        timeout=30,
    )
    return {
        "returncode": resultado.returncode,
        "stdout": resultado.stdout,
        "stderr": resultado.stderr,
    }


def salvar_json_saida(nome_arquivo, conteudo):
    caminho = OUTPUT_DIR / nome_arquivo
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(json.dumps(conteudo, ensure_ascii=False, indent=2), encoding="utf-8")
    return caminho.relative_to(PROJECT_ROOT).as_posix()

DOCUMENTOS = montar_corpus()
INDICE = criar_indice_tfidf(DOCUMENTOS)
print(f"Documentos indexados: {len(DOCUMENTOS)}")

Documentos indexados: 24


## Indexação de evidências e ferramentas locais

Esta etapa constrói a base de conhecimento que o agente poderá consultar.
O fluxo é o seguinte:

1. varremos `data/` e `target_project/` para coletar arquivos relevantes;
2. transformamos os textos em vetores TF-IDF;
3. calculamos similaridade para recuperar os trechos mais úteis para cada consulta.

Além da busca semântica simples com TF-IDF, a célula também define funções utilitárias que serão expostas como tools:
- listar tickets;
- ler arquivo por caminho relativo;
- listar arquivos do projeto-alvo;
- executar testes no `target_project`;
- salvar saídas em JSON.

Repare no padrão: em vez de dar acesso irrestrito ao sistema de arquivos, criamos funções controladas.
Isso melhora a auditabilidade e reduz o risco operacional.

In [4]:
from openai import OpenAI


def carregar_openai_api_key_de_arquivo():
    candidatos = [
        (Path.cwd() / ".env").resolve(),
        (PROJECT_ROOT / ".env").resolve(),
        (PROJECT_ROOT.parent / ".env").resolve(),
    ]
    vistos = set()

    for caminho in candidatos:
        if caminho in vistos:
            continue
        vistos.add(caminho)

        if not caminho.exists():
            continue

        for linha in caminho.read_text(encoding="utf-8").splitlines():
            conteudo = linha.strip()
            if not conteudo or conteudo.startswith("#") or "=" not in conteudo:
                continue

            chave, valor = conteudo.split("=", 1)
            if chave.strip() != "OPENAI_API_KEY":
                continue

            valor = valor.strip()
            if (valor.startswith('"') and valor.endswith('"')) or (valor.startswith("'") and valor.endswith("'")):
                valor = valor[1:-1]

            if valor:
                os.environ["OPENAI_API_KEY"] = valor
                return caminho.as_posix()

    return None


def exigir_openai_api_key():
    if os.getenv("OPENAI_API_KEY"):
        return

    caminho_carregado = carregar_openai_api_key_de_arquivo()
    if os.getenv("OPENAI_API_KEY"):
        print(f"OPENAI_API_KEY carregada de: {caminho_carregado}")
        return

    caminhos_tentados = [
        (Path.cwd() / ".env").resolve().as_posix(),
        (PROJECT_ROOT / ".env").resolve().as_posix(),
        (PROJECT_ROOT.parent / ".env").resolve().as_posix(),
    ]
    raise RuntimeError(
        "OPENAI_API_KEY não encontrada. Defina a variável de ambiente antes de executar as células com agentes reais, "
        "ou crie um arquivo .env com a linha OPENAI_API_KEY=... em um dos caminhos abaixo:\n"
        + "\n".join([f"- {c}" for c in caminhos_tentados])
    )


def parse_json_seguro(texto):
    try:
        return json.loads(texto)
    except Exception:
        inicio = texto.find("{")
        fim = texto.rfind("}")
        if inicio >= 0 and fim > inicio:
            return json.loads(texto[inicio:fim+1])
        raise


def schema_ferramentas_openai():
    return [
        {
            "type": "function",
            "name": "listar_tickets",
            "description": "Lista os tickets disponíveis para investigação.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
        {
            "type": "function",
            "name": "ler_arquivo",
            "description": "Lê um arquivo do projeto pelo caminho relativo.",
            "parameters": {
                "type": "object",
                "properties": {"caminho_relativo": {"type": "string"}},
                "required": ["caminho_relativo"],
                "additionalProperties": False,
            },
        },
        {
            "type": "function",
            "name": "buscar_evidencias",
            "description": "Busca evidências em tickets, logs, runbooks, histórico, políticas e código do target_project usando TF-IDF.",
            "parameters": {
                "type": "object",
                "properties": {
                    "consulta": {"type": "string"},
                    "k": {"type": "integer", "minimum": 1, "maximum": 8},
                },
                "required": ["consulta"],
                "additionalProperties": False,
            },
        },
        {
            "type": "function",
            "name": "listar_arquivos_do_target_project",
            "description": "Lista arquivos disponíveis no projeto-alvo que pode conter o código afetado pelo incidente.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
        {
            "type": "function",
            "name": "executar_testes_target_project",
            "description": "Executa a suíte de testes do target_project para observar o estado atual do projeto.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
    ]


def executar_tool_por_nome(nome, argumentos):
    if nome == "listar_tickets":
        return listar_tickets()
    if nome == "ler_arquivo":
        return carregar_arquivo_relativo(argumentos["caminho_relativo"] )
    if nome == "buscar_evidencias":
        return buscar_no_indice(INDICE, argumentos["consulta"], argumentos.get("k", 5))
    if nome == "listar_arquivos_do_target_project":
        return listar_arquivos_do_target_project()
    if nome == "executar_testes_target_project":
        return executar_testes_target_project()
    return {"erro": f"Tool desconhecida: {nome}"}


def executar_agente_openai(mensagem_usuario, instrucoes_sistema, modelo="gpt-4.1-mini", max_turnos=8):
    exigir_openai_api_key()
    cliente = OpenAI()
    historico = [{"role": "user", "content": mensagem_usuario}]
    trace = []

    for turno in range(max_turnos):
        resposta = cliente.responses.create(
            model=modelo,
            instructions=instrucoes_sistema,
            input=historico,
            tools=schema_ferramentas_openai(),
            tool_choice="auto",
        )

        chamadas = [item for item in resposta.output if item.type == "function_call"]
        textos = [item for item in resposta.output if item.type == "message"]

        if not chamadas:
            texto_final = "\n".join([c.text for msg in textos for c in msg.content if c.type == "output_text"])
            return {"final": texto_final, "trace": trace}

        historico += resposta.output

        for chamada in chamadas:
            argumentos = json.loads(chamada.arguments or "{}")
            resultado = executar_tool_por_nome(chamada.name, argumentos)
            trace.append({"turno": turno + 1, "tool": chamada.name, "arguments": argumentos, "result_preview": str(resultado)[:800]})
            historico.append({
                "type": "function_call_output",
                "call_id": chamada.call_id,
                "output": json.dumps(resultado, ensure_ascii=False),
            })

    raise RuntimeError("O agente excedeu o número máximo de turnos de uso de ferramentas.")

## Orquestração do agente com Function Calling

Agora conectamos o modelo às tools definidas antes.
O desenho desta integração segue quatro decisões didáticas importantes:

1. **pré-condição explícita**: sem `OPENAI_API_KEY`, a execução falha com mensagem clara;
2. **contrato de tools**: cada função é descrita com schema JSON (nome, descrição, parâmetros);
3. **laço de turnos**: o agente pode chamar tools em múltiplas rodadas, até produzir resposta final;
4. **rastreabilidade**: cada chamada é registrada em `trace` para auditoria posterior.

A função `parse_json_seguro` também adiciona robustez quando a saída do modelo vem com texto extra fora do JSON.
Esse tipo de defesa é importante em sistemas reais, porque reduz falhas por formatação imperfeita.

In [5]:
CONTRATO_SAIDA = {
    "incident_id": "identificador do incidente investigado",
    "diagnostico": {
        "causa_provavel": "texto curto",
        "confianca": "baixa | media | alta",
        "evidencias": ["lista de caminhos de arquivos ou trechos usados"],
    },
    "modulos_afetados": ["arquivos do target_project possivelmente afetados"],
    "proposta_conceitual": {
        "descricao": "alteração sugerida em linguagem natural",
        "tipo_mudanca": "validacao | transformacao | logging | teste | outro",
        "nao_executar_automaticamente": True,
    },
    "plano_de_testes": ["testes que deveriam ser adicionados ou executados"],
    "decisao": {
        "pode_preparar_patch_na_aula_4": "sim | nao",
        "motivo": "justificativa curta",
        "requer_revisao_humana": True,
    },
}

## Definindo o contrato de saída

Antes de executar o agente, definimos o formato mínimo esperado da resposta (`CONTRATO_SAIDA`).
Essa estrutura funciona como uma especificação de interface entre etapas do pipeline.

Por que isso é importante?
- facilita validação automática;
- reduz ambiguidades na interpretação da resposta;
- permite evolução incremental do sistema sem quebrar integrações.

Observe que o contrato já inclui restrições de segurança de processo, como:
- `nao_executar_automaticamente = True`;
- `requer_revisao_humana = True`.

Assim, o próprio formato da saída reforça governança técnica.

## Funções de validação determinística

Mesmo usando agentes com tools, a decisão de avanço precisa de regras objetivas.
Nesta seção, transformamos critérios de qualidade em verificações de código.

A lógica da validação cobre perguntas essenciais:
- o incidente foi identificado (`incident_id`)?
- há evidências citadas?
- os módulos afetados existem de fato no projeto-alvo?
- existe plano de testes?
- a proposta exige revisão humana e evita execução automática?

Perceba o ganho arquitetural: o agente pode ser criativo na análise,
mas a aprovação final depende de critérios estáveis, reprodutíveis e auditáveis.

Esse padrão (IA para hipótese + código para guarda de segurança) é um dos mais úteis em aplicações corporativas.

In [6]:
def normalizar_lista(valor):
    if isinstance(valor, list):
        return valor
    if valor in [None, ""]:
        return []
    return [valor]


def validar_artefato_proposta(artefato):
    problemas = []

    if not artefato.get("incident_id"):
        problemas.append("incident_id ausente")

    diagnostico = artefato.get("diagnostico", {})
    evidencias = normalizar_lista(diagnostico.get("evidencias"))
    if len(evidencias) == 0:
        problemas.append("nenhuma evidência informada")

    modulos = normalizar_lista(artefato.get("modulos_afetados"))
    arquivos_existentes = set(listar_arquivos_do_target_project())
    for modulo in modulos:
        if modulo not in arquivos_existentes and not any(path.endswith(modulo) for path in arquivos_existentes):
            problemas.append(f"módulo afetado não encontrado no target_project: {modulo}")

    plano = normalizar_lista(artefato.get("plano_de_testes"))
    if len(plano) == 0:
        problemas.append("plano_de_testes ausente")

    decisao = artefato.get("decisao", {})
    if decisao.get("requer_revisao_humana") is not True:
        problemas.append("a proposta deve exigir revisão humana")

    proposta = artefato.get("proposta_conceitual", {})
    if proposta.get("nao_executar_automaticamente") is not True:
        problemas.append("a proposta deve declarar que não será executada automaticamente")

    return {
        "aprovado_para_preparacao_de_patch": len(problemas) == 0,
        "problemas": problemas,
    }

## Agente revisor com tools

Agora o agente é colocado em ação com um objetivo concreto: investigar o incidente `INC-003`
e gerar uma proposta conceitual compatível com o contrato definido.

Nesta execução, vale observar três elementos:
1. as instruções de sistema delimitam claramente o que o agente pode e não pode fazer;
2. a mensagem de usuário define o escopo da tarefa;
3. o resultado final será tratado como dado estruturado, não como texto livre.

Em um cenário real, esse padrão ajuda a evitar deriva de escopo e respostas genéricas sem evidência.

In [7]:
instrucoes_revisor = f"""
Você é um agente revisor técnico para um sistema agêntico de investigação de incidentes.

Você deve usar tools para consultar tickets, logs, runbooks, histórico, políticas e código do target_project.
Não invente evidências. Não altere arquivos. Não gere patch executável.

Seu objetivo é produzir uma proposta conceitual que possa ser validada por código.
Responda apenas com JSON válido seguindo esta estrutura conceitual:
{json.dumps(CONTRATO_SAIDA, ensure_ascii=False, indent=2)}
"""

mensagem_revisor = """
Investigue o ticket INC-003 e produza uma proposta conceitual de correção.
A proposta deve estar pronta para passar por validação determinística.
"""

resultado_revisor = executar_agente_openai(mensagem_revisor, instrucoes_revisor)
print(resultado_revisor["final"])

OPENAI_API_KEY carregada de: C:/Users/cstefano/Desktop/Carlos/Alura/Skills and Go/Repositório/alura-skills-and-go-agentic-engineering/Aula 2/.env
{
  "incident_id": "INC-003",
  "diagnostico": {
    "causa_provavel": "Erro na transformação devido ao formato não ISO do campo created_at vindo no padrão brasileiro (DD/MM/YYYY HH:MM:SS) não suportado por fromisoformat.",
    "confianca": "alta",
    "evidencias": [
      "data/tickets/INC-003-date-format.md",
      "data/logs/orders_pipeline_2026-07-10.log",
      "target_project/data_samples/pedidos_com_data_regional.json",
      "target_project/src/mini_orders_pipeline/transformacao_pedidos.py"
    ]
  },
  "modulos_afetados": [
    "target_project/src/mini_orders_pipeline/transformacao_pedidos.py"
  ],
  "proposta_conceitual": {
    "descricao": "Alterar a função transformar_pedido para identificar se o campo created_at está no formato brasileiro (DD/MM/YYYY HH:MM:SS) e convertê-lo corretamente para o formato datetime antes da transform

## Validando a saída do agente

Aqui ocorre a transição mais importante do notebook:
a resposta do modelo deixa de ser apenas texto e passa por checagem formal.

Passos da célula a seguir:
1. tentar interpretar a saída como JSON (`parse_json_seguro`);
2. aplicar as regras determinísticas (`validar_artefato_proposta`);
3. inspecionar o resultado da validação.

Se houver problemas, isso não significa que a investigação falhou por completo.
Significa que o artefato ainda não atende o padrão mínimo para avançar com segurança.

In [8]:
artefato_revisor = parse_json_seguro(resultado_revisor["final"])
validacao = validar_artefato_proposta(artefato_revisor)
print(json.dumps(validacao, ensure_ascii=False, indent=2))

{
  "aprovado_para_preparacao_de_patch": true,
  "problemas": []
}


## Salvando artefato e validação

A persistência em arquivo tem duas funções didáticas e práticas:
- cria histórico reprodutível da execução;
- permite que outras etapas do pipeline consumam o resultado sem depender do estado da sessão.

O JSON salvo combina:
- proposta conceitual do agente;
- resultado da validação determinística;
- rastros das tools utilizadas.

Com isso, você tem um artefato auditável para análise posterior, discussão em revisão técnica
ou preparação de uma etapa futura de implementação.

In [9]:
saida = {
    "proposta": artefato_revisor,
    "validacao": validacao,
    "trace_tools": resultado_revisor["trace"],
}

caminho = salvar_json_saida("proposals/incidente_inc003_proposta_validada.json", saida)
print("Arquivo salvo em:", caminho)

Arquivo salvo em: outputs/proposals/incidente_inc003_proposta_validada.json


## Fechamento da Aula 2

Ao concluir este notebook, você construiu um fluxo completo de investigação assistida por agentes com controle de qualidade por código.
Em termos de arquitetura, o sistema final:

- usa agentes reais com acesso controlado a ferramentas;
- consulta evidências externas e arquivos do projeto-alvo sob demanda;
- registra rastros de execução para auditoria;
- exige contrato estruturado de saída;
- aplica validações determinísticas antes de qualquer evolução operacional.

A principal aprendizagem é estratégica:
modelos de linguagem são excelentes para síntese e hipótese,
mas a confiabilidade do processo vem de regras explícitas, validação e revisão humana.

Com essa base, você está pronto para evoluir o fluxo para etapas mais operacionais em módulos seguintes,
como geração de relatório técnico, proposta de mudanças e planejamento de testes com maior detalhamento.